In [6]:
from langgraph.checkpoint.memory import MemorySaver  # Para memoria a nivel de hilo
from langgraph.store.memory import InMemoryStore  # Para memoria a largo plazo (usuario)
from langgraph.graph.message import AnyMessage, add_messages  # Para gestionar mensajes en el estado del grafo
from langgraph.managed.is_last_step import RemainingSteps  # Para rastrear el límite de recursión

# Inicializar la memoria a largo plazo
in_memory_store = InMemoryStore()

# Inicializar el checkpointer para memoria a nivel de hilo
checkpointer = MemorySaver()


In [7]:
from dotenv import load_dotenv
import openai
import os
from langchain_core.tools import tool
from extract_to_cmaps import prompt_extract_cmapss, generate_cmapss_assistant_prompt
import json
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
# Carga las variables de entorno desde el archivo .env
load_dotenv(dotenv_path=".env", override=True)

# Recupera la clave API de OpenAI desde las variables de entorno
openai.api_key = os.getenv("OPENAI_API_KEY")

# Verifica que la clave API se haya cargado correctamente
if openai.api_key is None:
    raise ValueError("La clave API de OpenAI no está configurada correctamente.")

# Ahora puedes usar la clave API en LangChain o directamente con OpenAI
llm = ChatOpenAI(temperature=0, model="gpt-3.5-turbo")  # O el modelo que prefieras


In [8]:
from typing_extensions import TypedDict
from typing import Annotated, List

class State(TypedDict):
    user_id: str
    messages: Annotated[list[AnyMessage], add_messages]
    
    # Historial de mensajes
    messages: Annotated[list[AnyMessage], add_messages]
    # loaded_memory: Stores information loaded from the long-term memory store, 
    # typically user preferences or historical context.
    loaded_memory: str
    # remaining_steps: Used by LangGraph to track the number of allowed steps 
    # to prevent infinite loops in cyclic graphs.
    remaining_steps: RemainingSteps 


In [9]:

from langchain_core.tools import tool # Decorator to define a function as a LangChain tool
import ast # Module to safely evaluate strings containing Python literal structures
from extract_to_cmaps import prompt_extract_cmapss, generate_cmapss_assistant_prompt

# @tool
# def extract_cmapss_data(message: str):
#     """
#     Esta herramienta extrae los datos estructurados para alimentar un modelo de predicción RUL basado en CMAPSS.
#     """
#     print("\nRecibiendo mensaje:", message)  # Verifica que el mensaje llega
#     prompt = prompt_extract_cmapss(message)
#     print("\nPrompt generado:", prompt)  # Verifica que el prompt se genera correctamente
    
#     # Llamada al modelo
#     response = llm(prompt)
#     print("\nRespuesta del modelo:", response)  # Verifica lo que devuelve el modelo
    
#     try:
#         # Intentamos parsear la respuesta para asegurar que está en formato JSON
#         parsed_response = json.loads(response)
#     except json.JSONDecodeError:
#         # Si la respuesta no es un JSON válido, devolver un mensaje de error
#         parsed_response = {
#             "error": "La respuesta del modelo no es un JSON válido",
#             "modelo_seleccionado": "FD001"  # Selección por defecto
#         }
#     return json.dumps(parsed_response)

@tool
def extract_cmapss_data(message: str):
    """
    Extraer datos del estado de un motor aeronáutico para alimentar un modelo de predicción RUL basado en CMAPSS.
    """
    print(f"Extrayendo datos del mensaje: {message}")
    
    # Simulamos la extracción de los datos
    extracted_data = {
        "unidad": 200,
        "tiempo_ciclos": 150,
        "configuraciones_operativas": [1, 2, 3],
        "mediciones_sensores": {
            "s_1": 0.25, "s_2": 0.35, "s_3": 0.45, "s_7": 550
        },
        "modelo_seleccionado": "FD001"
    }
    
    return extracted_data



In [10]:
cmaps_tools = [extract_cmapss_data]

llm_with_tools = llm.bind_tools(tools)

In [11]:
from langgraph.prebuilt import ToolNode # Pre-built node for executing tools

cmapss_tool_node = ToolNode(cmaps_tools)

In [ ]:
from langchain_core.messages import ToolMessage, SystemMessage, HumanMessage # Message types for conversation history
from langchain_core.runnables import RunnableConfig # For configuration parameters passed to runnables.

def generate_cmapss_assistant_prompt(memory: str = "None") -> str:
    return f"""
    Eres un asistente especializado en la extracción de datos sobre el estado de motores aeronáuticos, en particular para alimentar modelos de predicción RUL basados en CMAPSS.
    Si no encuentras información sobre los sensores o datos específicos, es importante que no inventes ni rellenes datos. 
    Usa únicamente los datos mencionados por el usuario.
    
    Recuerda: el modelo seleccionado dependerá de las condiciones mencionadas (nivel del mar, fan degradation, etc).
    
    Mensaje del usuario: {memory}
    """

def cmapss_assistant(state: State, config = RunnableConfig)
